In [14]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../train.csv")

X = df.drop(columns=["Price_INR_Lakhs", "Property_ID"])
y = df["Price_INR_Lakhs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

categorical_features = [
    "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)],
    sparse_threshold=0
)


### Why the R2 score did not match notebook 25

**Previous code (bug):**

```python
for setting in settings:
    model = Pipeline([
        ("preprocessor",preprocessor),
        ("regressor",HistGradientBoostingRegressor())   # empty brackets
    ]
    )
```

1. `setting` was never passed in, so every run used the default parameters.
2. There was no `random_state`. With more than 10,000 training rows, early stopping turns on automatically and holds back a **random** 10% of the training data, so each run stopped at a different number of trees and gave a slightly different R2 (for example 0.7870 one run, 0.7857 the next).

**Fix:** pass `**setting` and `random_state=42`, the same seed as notebook 25.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

settings = [
    {"learning_rate":0.03,"max_iter":100},
    {"learning_rate":0.05,"max_iter":100},
    {"learning_rate":0.10,"max_iter":100},
    {"learning_rate":0.05,"max_iter":200},
    {"learning_rate":0.10,"max_iter":200}
]

results=[]

for setting in settings:
    model = Pipeline([
        ("preprocessor",preprocessor),
        ("regressor",HistGradientBoostingRegressor(
            **setting,
            max_leaf_nodes=31,
            random_state=42
        ))
    ])

    model.fit(X_train,y_train)

    y_pred = model.predict(X_val)

    results.append({
        **setting,
        "MAE": mean_absolute_error(y_val,y_pred),
        "RMSE": mean_squared_error(y_val,y_pred) ** 0.5,
        "R2": r2_score(y_val,y_pred),
        "trees_used": model.named_steps["regressor"].n_iter_
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
results_df

,learning_rate,max_iter,MAE,RMSE,R2,trees_used
2,0.10,100,32.846547,68.672493,0.786996,56
4,0.10,200,32.846547,68.672493,0.786996,56
3,0.05,200,32.834453,68.763963,0.786428,112
1,0.05,100,32.972217,68.873676,0.785746,100
0,0.03,100,34.729663,70.156013,0.777694,100
